# Sprint 2 — Nikita sparse TF-IDF sweep

Attach the `llm-classification-finetuning` competition data, grant this notebook access to the five `CLEARML_*` Kaggle Secrets, select a CPU session, enable Internet for the Git clone and package installation, then use **Save Version → Save & Run All**. Model selection uses frozen fold 7, not the Kaggle public leaderboard.


In [ ]:
from pathlib import Path

REPOSITORY = 'https://github.com/kujifined/PMLDL-llm-classification-finetuning.git'
BRANCH = 'experiment/E202609140001-sparse-tfidf-sweep'
REPO_DIR = Path('/kaggle/working/team-repo')
DATA_DIR = Path('/kaggle/input/llm-classification-finetuning')
assert DATA_DIR.is_dir(), 'Attach the competition data before running.'


In [ ]:
if not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPOSITORY} {REPO_DIR}
%cd {REPO_DIR}
!git status --short --branch


In [ ]:
!python -m pip install -q -r requirements-baseline.lock 'clearml>=1.17,<3'
import sklearn, numpy, pandas, scipy
print('scikit-learn', sklearn.__version__)
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('scipy', scipy.__version__)
!git status --short --branch


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secret_names = (
    'CLEARML_API_ACCESS_KEY',
    'CLEARML_API_SECRET_KEY',
    'CLEARML_API_HOST',
    'CLEARML_WEB_HOST',
    'CLEARML_FILES_HOST',
)
secret_client = UserSecretsClient()
missing = []
for secret_name in secret_names:
    try:
        os.environ[secret_name] = secret_client.get_secret(secret_name)
    except Exception:
        missing.append(secret_name)
assert not missing, f'Missing or ungranted Kaggle Secrets: {missing}'
print('ClearML credentials loaded from Kaggle Secrets (values hidden).')


In [ ]:
!PYTHONPATH=src python -u scripts/sweep_sparse_sprint2.py --data-dir {DATA_DIR} --external-data


In [ ]:
import json
import shutil

run_dirs = sorted(
    (REPO_DIR / 'results' / 'runs').glob('E202609140001__*'),
    key=lambda path: path.stat().st_mtime,
)
assert run_dirs, 'No Sprint 2 run directory was produced.'
run_dir = run_dirs[-1]
run_id = run_dir.name
metrics = json.loads((run_dir / 'metrics.json').read_text())
run_metadata = json.loads((run_dir / 'run.json').read_text())
assert metrics['status'] == 'completed', metrics
artifact_dir = REPO_DIR / 'artifacts' / run_id
bundle_dir = Path('/kaggle/working/nikita_sprint2_handoff')
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
shutil.copytree(run_dir, bundle_dir / 'results' / 'runs' / run_id)
shutil.copytree(artifact_dir, bundle_dir / 'artifacts' / run_id)
archive = shutil.make_archive(str(bundle_dir), 'zip', bundle_dir)
print('Run:', run_id)
print('Validation:', metrics['summary']['validation'])
task_id = run_metadata['tracking']['task_id']
if task_id:
    from clearml import Task
    print('ClearML:', Task.get_task_output_log_web_page(task_id))
else:
    print('ClearML tracking failed:', run_metadata['tracking']['error'])
print('Download:', archive)
